# Task 1 — Player Discipline: Yellow Cards by Position

## Analytic question formulation

**On average, do midfielders pick up more yellow cards per match played than
defenders at the 2026 FIFA World Cup?**

Discipline (cards) is often assumed to track with tackling/defensive duels,
but modern World Cups also feature heavy midfield pressing battles. This
task investigates whether the *rate* of yellow cards per appearance
(cards are normalised by appearances so that squad players who featured in
more matches are not automatically penalised) differs meaningfully between
the two positional groups.

We restrict the population to outfield players (**defenders, DF**, and
**midfielders, MF**) who made **at least one appearance** in the tournament
— non-playing squad members carry no discipline information and are
excluded, exactly as in the worked example.

**Skills demonstrated:** data wrangling → sampling → descriptive statistics
→ confidence interval → two-sample *t*-test.

In [ ]:
import sys
sys.path.append("../src")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from data_prep import load_players
from stats_utils import describe, ci_mean, two_sample_ttest, levene_test

np.random.seed(42)
pd.set_option("display.max_columns", None)

players = load_players()
players.head()

## Data wrangling

`players_raw.csv` (compiled from FIFA.com / fbref.com match & disciplinary
reports) holds one row per player who featured for one of the 48 squads,
with appearances, minutes, goals, and cards for the tournament.
`data_prep.load_players()`:

1. Normalises team-name spelling across sources.
2. Coerces numeric columns and **keeps only players with `appearances >= 1`**
   (never-used squad members carry no discipline signal and are dropped,
   exactly as in the worked example).
3. Derives `cards_per_app = yellow_cards / appearances`, our unit of
   analysis — a *rate* rather than a raw count, so that a player who played
   all 7 matches for a finalist is compared fairly with one who played a
   single group match.

We now isolate the two positional groups under study.

In [ ]:
outfield = players[players["position"].isin(["DF", "MF"])].copy()
print("Players with >=1 appearance, DF or MF:", outfield.shape[0])
print(outfield["position"].value_counts())
outfield[["player_name", "team", "position", "appearances", "yellow_cards", "cards_per_app"]].sample(
    8, random_state=1
)

## Data preparation and sampling

**Population:** every defender and every midfielder who made at least one
appearance for their national team at the 2026 World Cup (two sub-populations,
one per position).

**Variable of interest:** `cards_per_app` (yellow cards ÷ appearances) — a
continuous rate variable, one observation per player.

**Sampling technique:** although we were able to compile disciplinary
records for the whole population, real-world data collection is rarely
perfectly complete (missing box-score detail, unverified benched minutes,
etc.). To mimic a realistic, defensible inferential workflow — and to keep
the two groups on an equal footing for the *t*-test — we draw a **simple
random sample (SRS), without replacement, of up to 45 players per
position** from the compiled population, using a fixed random seed for
reproducibility. This comfortably clears the *n* ≥ 30 rule of thumb needed
for the sampling distribution of the mean to be approximately normal via
the Central Limit Theorem, even though the underlying `cards_per_app`
distribution is right-skewed (many players with zero cards).

In [ ]:
def srs(df, n, seed=42):
    n = min(n, len(df))
    return df.sample(n=n, random_state=seed, replace=False)

mf_pop = outfield[outfield["position"] == "MF"]
df_pop = outfield[outfield["position"] == "DF"]

mf_sample = srs(mf_pop, 45)
df_sample = srs(df_pop, 45)

print(f"Midfielder population: {len(mf_pop)}  -> sample n={len(mf_sample)}")
print(f"Defender population:   {len(df_pop)}  -> sample n={len(df_sample)}")

## Descriptive statistics

In [ ]:
desc_table = pd.DataFrame(
    {"Midfielders": describe(mf_sample["cards_per_app"]), "Defenders": describe(df_sample["cards_per_app"])}
)
desc_table

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].hist(mf_sample["cards_per_app"], bins=10, color="#4C72B0", edgecolor="white")
ax[0].set_title("Midfielders: cards per appearance")
ax[1].hist(df_sample["cards_per_app"], bins=10, color="#DD8452", edgecolor="white")
ax[1].set_title("Defenders: cards per appearance")
for a in ax:
    a.set_xlabel("Yellow cards / appearance")
    a.set_ylabel("Count")
plt.tight_layout()
plt.savefig("../report/figs/task1_hist.png", dpi=120)
plt.show()

desc_table

## Inferential statistics — confidence interval

We estimate a 95% confidence interval for the **population mean
cards-per-appearance of midfielders** (the group the analytic question is
centred on), using the *t*-distribution (population SD unknown, estimated
from the sample).

In [ ]:
ci = ci_mean(mf_sample["cards_per_app"], confidence=0.95)
print(f"n = {ci['n']}")
print(f"Sample mean cards/appearance (MF) = {ci['mean']:.4f}")
print(f"95% CI: ({ci['ci_low']:.4f}, {ci['ci_high']:.4f})")
ci

## Inferential statistics — two-sample *t*-test

$H_0: \mu_{MF} = \mu_{DF}$ (mean cards/appearance equal between midfielders
and defenders)

$H_1: \mu_{MF} \neq \mu_{DF}$

We first check the equal-variance assumption with Levene's test, then run
Welch's two-sample *t*-test (robust regardless of the outcome), at
$\alpha = 0.05$.

In [ ]:
lev = levene_test(mf_sample["cards_per_app"], df_sample["cards_per_app"])
print("Levene's test p-value:", lev["p_value"])

result = two_sample_ttest(
    mf_sample["cards_per_app"], df_sample["cards_per_app"], equal_var=lev["p_value"] > 0.05
)
for k, v in result.items():
    print(f"{k}: {v}")

alpha = 0.05
if result["p_value"] < alpha:
    print(f"\nReject H0 (p={result['p_value']:.4f} < {alpha}): means differ significantly.")
else:
    print(f"\nFail to reject H0 (p={result['p_value']:.4f} >= {alpha}): no significant difference detected.")

## Conclusion

*(Auto-filled after running the cells above with real data — see the
printed sample means, CI bounds, and t-test p-value. Summarize here: the
direction and magnitude of the difference in cards/appearance between
midfielders and defenders, whether it is statistically significant at
α=0.05, and how the CI for midfielders' mean cards/appearance should be
interpreted.)*